# 算子工程化开发

本节围绕 AddCustom 完成一条自定义算子工程化链路：原型定义、Host 侧 Tiling、Kernel 侧实现、CMake 编译打包、部署和 aclnn 单算子 API 调用验证。扩展概念集中在课后练习之后的“扩展阅读”。

### 学习前置要求

- 已完成 Ascend C 核函数开发基础课程
- 理解 Tiling 机制基本概念
- 了解 CMake 基础用法
- 环境已安装 CANN Toolkit >= 9.0.0
- CMake >= 3.16、g++ >= 7.5、glibc >= 2.31；若直接使用 `cmake --preset`，建议 CMake >= 3.19

### 学习目标

- 使用 msopgen 创建自定义算子工程
- 掌握 Host 侧 Tiling 实现
- 掌握 Kernel 侧工程化实现
- 理解算子工程的 CMake 编译组织和主要构建产物
- 完成算子编译部署和 aclnn 调用验证

### 本节内容

- 概述与环境准备：理解工程化开发概念，完成环境自检
- 设计：从数学公式识别输入输出，编写 JSON 原型，用 msopgen 创建工程并解读 OpDef
- Host 侧实现：定义 TilingData，实现 TilingFunc，设置 BlockDim 和 workspace
- Kernel 侧实现：注册并解析 TilingData，使用信息宏适配类型，编写工程化 Kernel
- 编译部署：通过 build.sh 生成 .run 包，理解 CMake 编译组织，部署并配置 set_env.bash
- 调用验证：使用 aclnn 两段式接口验证算子功能
- 练习：完成 SubCustom（减法算子）的开发，巩固所学
- 扩展阅读：属性传递（E1）、多分支（E2）、运行时加载（E4）、算子包部署细节与优先级（E5）、交叉编译（E6）、验证工程 CMakeLists.txt 配置（E7）、高阶 API 配套 Tiling（E8），按需深入


## 1. 概述与环境准备

### 1.1 工程化算子开发概述

工程化算子开发是基于 msopgen 生成的自定义算子工程，完成原型定义、Host 侧 Tiling、Kernel 侧计算、编译部署和单算子 API 调用的开发方式，适用于将自定义算子接入 CANN 框架运行。

**集成方式**：CANN 算子支持单算子API调用（aclnn接口）、算子入图、AI框架调用。本节聚焦单算子API调用，这是验证算子最直接的方式。


### 1.2 端到端流程

<img src="./images/operator_implementation_flow.png"  alt="operator_implementation_flow" />


### 1.3 环境初始化

导入 CANN 环境变量并创建代码目录：


In [ ]:
!mkdir -p Sources/06.03

import os, subprocess
from pathlib import Path

Path("Sources/06.03").mkdir(exist_ok=True)

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True, text=True, check=True
) 
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")


In [ ]:
# 环境自检
import subprocess, os

checks = [
    ('CMake版本', 'cmake --version', '版本 >= 3.16'),
    ('GCC版本', 'g++ --version', '版本 >= 7.5'),
    ('glibc版本', 'ldd --version', '版本 >= 2.31'),
    ('环境变量', 'echo $ASCEND_TOOLKIT_HOME', '非空，指向CANN安装路径'),
    ('环境变量', 'echo $ASCEND_HOME_PATH', '非空，指向CANN安装路径'),
]

print('=== 环境自检清单 ===')
for name, cmd, expected in checks:
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        output = result.stdout.strip()[:100]
        status = '[OK]' if result.returncode == 0 else '[FAIL]'
        print(f'{status} {name}: {output}')
        print(f'   预期: {expected}')
    except Exception as e:
        print(f'[FAIL] {name}: 执行失败 - {e}')
        print(f'   预期: {expected}')
    print()


---

## 2. 设计：算子功能设计与原型定义

工程化开发的第一步是设计，需要明确算子做什么，产出 JSON 原型定义，再由 msopgen 落地为工程代码。

### 2.1 从数学公式出发

从公式识别三类要素：输入张量、输出张量、属性（标量参数）。以 `z = x + y` 为例：两个输入 x/y、一个输出 z、无属性。

> 如果公式是 `z = x + alpha * y`，则 `alpha` 是属性，详见扩展阅读 E1。


### 2.2 认识 OpDef

**OpDef**（算子原型定义）描述了算子的输入输出、属性和实现配置（如关联的 Tiling 实现）。框架根据 OpDef 自动生成 aclnn 接口和 InferShape，是后续自动化的基础。

开发流程中 OpDef 经历 JSON→C++ 的转化：开发者编写 JSON 原型定义文件，msopgen 自动生成 `class XxxCustom : public ge::OpDef` C++ 注册代码。


### 2.3 编写 JSON 原型定义

关键信息：

1. 算子名称：`"op"` 使用大驼峰（如 `AddCustom`），决定生成的核函数名和文件名
2. param_type：`required`（必须）、`optional`（可选）、`dynamic`（动态数量）
3. type/format：列表长度必须一致，一一对应
4. 芯片范围：通过 msopgen `-c` 参数指定


In [ ]:
%%writefile Sources/06.03/add_custom.json
[
    {
        "op": "AddCustom",
        "input_desc": [
            {
                "name": "x",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            },
            {
                "name": "y",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            }
        ],
        "output_desc": [
            {
                "name": "z",
                "param_type": "required",
                "format": ["ND", "ND"],
                "type": ["float16", "float"]
            }
        ]
    }
]


### 2.4 用 msopgen 创建算子工程

JSON 原型定义完成后，使用 **msopgen** 生成工程代码。核心参数：

- -i：JSON 原型定义文件路径
- -c：目标芯片型号（格式：`ai_core-ascend910b1`，`ai_core-ascend950`）
- -lan：编程语言，取值 `cpp`
- -out：输出目录

> -c 使用全小写格式（如 `ascend910b1`），msopgen 自动映射为框架统一标识并写入 CMakePresets.json。


**准备 msopgen**：CANN 自带 msopgen 工具，验证可用性：


In [ ]:
# CANN 9.0.0 已自带 msopgen 工具
!which msopgen && msopgen -h


**创建算子工程**：执行以下命令创建工程：


In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/06.03/custom_op

# 使用msopgen工具创建算子工程
!msopgen gen -i Sources/06.03/add_custom.json -c ai_core-ascend950 -lan cpp -out Sources/06.03/custom_op


### 2.5 工程目录结构

msopgen 生成的工程目录：

```
custom_op/
|-- build.sh                  // 编译脚本
|-- CMakePresets.json         // 编译配置（msopgen自动生成）
|-- op_host/add_custom.cpp    // [需修改] 算子原型注册 + Tiling实现
|-- op_kernel/
|   |-- add_custom.cpp        // [需修改] 算子代码实现
|   |-- add_custom_tiling.h   // [需修改] TilingData 定义
```

三个需修改的核心文件：`op_host/add_custom.cpp`（原型注册+Tiling实现）、`op_kernel/add_custom_tiling.h`（TilingData定义）、`op_kernel/add_custom.cpp`（Kernel实现）。

> TilingData 头文件必须放在 `op_kernel` 目录下，因为该目录文件会被打包进算子包供在线编译使用。


In [ ]:
!cd Sources/06.03/custom_op;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'


### 2.6 OpDef 代码解读

查看 msopgen 自动生成的 Host侧代码：


In [ ]:
!cat Sources/06.03/custom_op/op_host/add_custom.cpp


msopgen 自动生成的 OpDef 代码包含四个关键部分：

- **输入输出定义**：

    ```cpp
    this->Input("x").ParamType(REQUIRED).DataType({ge::DT_FLOAT16, ge::DT_FLOAT}).Format({ge::FORMAT_ND, ge::FORMAT_ND});
    ```

    DataType 和 Format 列表长度必须一致。

- **Tiling 注册**：

    ```cpp
    this->AICore().SetTiling(optiling::TilingFunc);  // 关联 Host侧 TilingFunc
    ```

- **芯片注册**：

    ```cpp
    this->AICore().AddConfig("ascend950");  // 支持的芯片型号
    ```

    这里的 `"ascend950"` 来自创建工程时 `msopgen -c ai_core-<soc_version>` 中的 `<soc_version>`，也就是该算子声明支持的 AI 处理器型号。

    msopgen 会把芯片型号写入 `CMakePresets.json` 的 `ASCEND_COMPUTE_UNIT`，并在生成的 OpDef 中通过 `AddConfig` 注册。本节命令使用 `-c ai_core-ascend950`，生成工程中的统一标识为 `ascend950`，因此这里看到的是 `AddConfig("ascend950")`。

- **InferShape / InferDataType 推导函数**：

    用于运行前确定输出的 shape 和 datatype。本例中输出 z 与输入 x 一致，所以 InferShape 设 `*outputShape = *inputShape`，InferDataType 设 `SetOutputDataType(0, GetInputDataType(0))`。

    对于简单场景，可用 **Follow 接口**简化：`this->Output("z").Follow("x")` 表示输出 z 的 datatype/format/shape 与输入 x 相同。Follow 是 InferShape 的简化写法，能用 Follow 表达的逻辑建议优先使用。


---

## 3. 实现：Host侧 Tiling 实现

### 3.1 Tiling 概念与切分策略

AI Core 的 Local Memory 容量有限，无法一次性容纳全部输入输出数据，因此需要将数据切分为小块，通过 "搬入→计算→搬出" 的流水线循环完成全量计算——这就是 **Tiling**。

<img src="./images/tile_animation_slow.gif" alt="tiling" width="700px" />


In [ ]:
import numpy as np

def tiling_array_add(arr1, arr2, tile_num, arr_length):
    result = np.zeros_like(arr1)
    tile_size = arr_length // tile_num
    print("开始Tiling分块相加...")
    for start_idx in range(0, arr_length, tile_size):
        end_idx = min(start_idx + tile_size, arr_length)
        tile1 = arr1[start_idx:end_idx]
        tile2 = arr2[start_idx:end_idx]
        tile_result = tile1 + tile2
        result[start_idx:end_idx] = tile_result
        print(f"  完成Tile：arr1[{start_idx}:{end_idx}] + arr2[{start_idx}:{end_idx}] = {tile_result}")
    print("Tiling分块相加完成！")
    return result

if __name__ == "__main__":
    max_ub = 10
    arr_length = 10
    input_output_num = 3
    max_once_num = max_ub // input_output_num
    tile_num = arr_length // max_once_num
    arr_a = np.arange(1, arr_length+1, dtype=np.int32)
    arr_b = np.full(arr_length, 5, dtype=np.int32)
    print("=" * 40)
    print("原始数组arr_a：", arr_a)
    print("原始数组arr_b：", arr_b)
    print("=" * 40)
    result_direct = arr_a + arr_b
    print("直接相加结果：", result_direct)
    print("=" * 40)
    result_tiling = tiling_array_add(arr_a, arr_b, tile_num, arr_length)
    print("=" * 40)
    print("Tiling分块结果：", result_tiling)
    print("=" * 40)


**多核切分与 tile 内分块**：

Ascend C 的 Tiling 机制包含两层切分：

- 多核切分（BlockDim）：将 `totalLength` 均分到多个 AI Core 并行计算。TilingFunc 中 `SetBlockDim(NUM_BLOCKS)` 设置逻辑 Block 数；Kernel侧 `GetBlockNum()` 获取总数、`GetBlockIdx()` 获取当前编号，每个 Block 处理 `totalLength / GetBlockNum()` 的数据。

- tile 内分块（tileNum）：每个 Block 内部再将数据切为 `tileNum` 块流水处理。`tileNum` 越大，单次搬运量越小。

**本例的切分策略**：

AddCustom 算子的输入为 shape `[8, 2048]` 的两个向量 x 和 y，总元素数 `totalLength = 16384`。切分思路：

1. 多核切分：设定 `blockDim = 8`，将 16384 个元素均分到 8 个 AI Core，每个 Core 处理 `blockLength = 16384 / 8 = 2048` 个元素。
2. tile 内不分块：设定 `tileNum = 1`，因为 2048 个 float16 元素（约 4KB）远小于 Local Memory 容量，无需进一步切分。每个 Core 一次性将 2048 个元素搬入 Local Memory，计算后搬出。

> 当 `tileNum > 1` 时，每个 Core 内部会将 `blockLength` 再切为 `tileNum` 个小块流水处理，减少单次搬运量。本例简化为 `tileNum = 1`，重点体现多核切分。

<img src="./images/aclnn_tiling_strategy_diagram.png" alt="tiling strategy" width="800px" />

切分参数通过 TilingData 从 Host 传递到 Kernel，下面看具体怎么写：


### 3.2 TilingData 定义

TilingData 是 Host 和 Kernel 之间的"契约"：Host侧 TilingFunc 写入切分参数，Kernel侧读取参数指导调度。推荐使用标准 C++ struct 定义，支持 bool 和数组，允许同名结构体隔离：


In [ ]:
%%writefile Sources/06.03/custom_op/op_kernel/add_custom_tiling.h
#ifndef ADD_CUSTOM_TILING_H
#define ADD_CUSTOM_TILING_H
#include <cstdint>

struct AddCustomTilingData {
    uint32_t totalLength;  // 总计算数据量
    uint32_t tileNum;      // 每个Block上总计算数据分块个数
};
#endif // ADD_CUSTOM_TILING_H


约束：仅支持 POD 类型，不支持成员函数、指针和引用；GetTilingData 不自动初始化，必须显式赋值。

> 另有宏定义方式（`BEGIN_TILING_DATA_DEF`），仅在使用高阶 API 时才需使用，详见扩展阅读 E8。


### 3.3 TilingFunc 实现

根据上面的切分策略，TilingFunc 需要完成以下工作：从输入 shape 计算出 `totalLength`，将切分参数写入 TilingData，设置 BlockDim 和 workspace。签名固定为 `ge::graphStatus TilingFunc(gert::TilingContext *context)`，关键步骤：

1. **获取 TilingData 指针**：`context->GetTilingData<AddCustomTilingData>()`
2. **获取输入 shape**：`context->GetInputShape(0)->GetOriginShape().GetShapeSize()`
3. **填写 TilingData**：`tiling->totalLength = totalLength; tiling->tileNum = TILE_NUM;`
4. **设置 BlockDim**：`context->SetBlockDim(NUM_BLOCKS)`
5. **设置 workspace**：`currentWorkspace[0] = 0`

**workspace 说明**：workspace 是算子执行期间可能需要的额外 Device 内存。AddCustom 不需要（计算过程只用 Local Memory），所以设为 0。需要大量中间暂存（如排序、矩阵乘法）时才分配非零 workspace。

**GetOriginShape vs GetStorageShape**：前者返回逻辑 shape（用户视角），后者返回物理存储 shape。ND 格式时两者一致，课后练习中使用了 GetStorageShape。

### 3.4 TilingFunc 完整实现


In [ ]:
%%writefile Sources/06.03/custom_op/op_host/add_custom.cpp
#include "../op_kernel/add_custom_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
const uint32_t NUM_BLOCKS = 8;
const uint32_t TILE_NUM = 1;

static ge::graphStatus TilingFunc(gert::TilingContext *context) {
    AddCustomTilingData *tiling = context->GetTilingData<AddCustomTilingData>();
    uint32_t totalLength = context->GetInputShape(0)->GetOriginShape().GetShapeSize();
    tiling->totalLength = totalLength;
    tiling->tileNum = TILE_NUM;
    context->SetBlockDim(NUM_BLOCKS);
    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = 0;
    return ge::GRAPH_SUCCESS;
}
}  // namespace optiling

namespace ge {
static graphStatus InferShape(gert::InferShapeContext *context)
{
    const gert::Shape *intputShape = context->GetInputShape(0);
    gert::Shape *outputShape = context->GetOutputShape(0);
    *outputShape = *intputShape;
    return GRAPH_SUCCESS;
}

static graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    context->SetOutputDataType(0, context->GetInputDataType(0));
    return ge::GRAPH_SUCCESS;
}
}  // namespace ge

namespace ops {
class AddCustom : public OpDef {
public:
    explicit AddCustom(const char *name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc)
            .AddConfig("ascend950");
    }
};
OP_ADD(AddCustom);
}  // namespace ops


---

## 4. 实现：Kernel侧工程化实现

### 4.1 核函数工程化改造要点

Kernel 侧代码的主体仍然是核函数课程中熟悉的 `Init -> Process -> CopyIn -> Compute -> CopyOut` 模式；工程化改造的重点，是让 Kernel 承接 Host 侧 TilingFunc 计算出的切分结果，而不是在 Kernel 中写死数据规模和分块常量。

从核函数直调迁移到算子工程时，重点检查以下 5 点：

- **入口签名由框架约定**：参数顺序固定为“输入 → 输出 → workspace → tiling”，不要手动调整。`workspace` 和 `tiling` 由框架在执行时传入。
- **TilingData 从 `tiling` 参数解析**：Host 侧把 `totalLength`、`tileNum` 等切分参数写入 TilingData，Kernel 侧通过 `GET_TILING_DATA(tilingData, tiling)` 读取，再传给算子类初始化。
- **标准 C++ TilingData 需要注册**：本课程用 `struct AddCustomTilingData` 定义 TilingData，因此 Kernel 入口先调用 `REGISTER_TILING_DEFAULT(AddCustomTilingData)`，声明默认 TilingData 类型，再解析数据。
- **多核切分由 Host 和 Kernel 配合完成**：Host 侧 `context->SetBlockDim(NUM_BLOCKS)` 设置逻辑 Block 数，Kernel 侧 `GetBlockNum()` 获取总数，`GetBlockIdx()` 获取当前逻辑核编号。本例按 `blockLength = totalLength / GetBlockNum()` 计算每个 Core 的数据段。
- **类型适配交给信息宏**：OpDef 支持 `float16` 和 `float`，Kernel 中用 `DTYPE_X`、`DTYPE_Y`、`DTYPE_Z` 表示当前 binary 对应的 C++ 类型。详细解释见 4.2。

AddCustom 的测试输入为 `[8, 2048]`，Host 侧设置 `BlockDim=8`、`tileNum=1`，所以每个 Core 处理 2048 个元素，并按“搬入 → 计算 → 搬出”执行一次循环。如果扩展到任意 shape，需要在 TilingFunc 或 Kernel 中补充余数、尾块和对齐处理。

Kernel 入口定义形式如下：

```cpp
extern "C" __global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y,
                                                 GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(AddCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelAdd op;
    op.Init(x, y, z, tilingData.totalLength, tilingData.tileNum);
    op.Process();
}
```

> `__global__` 表示设备侧执行；`__aicore__` 表示在 AI Core 上运行；`GM_ADDR` 是 Global Memory 地址宏。`workspace` 是框架按 TilingFunc 设置的 workspace size 传入的 GM 地址。本例 `currentWorkspace[0] = 0`，因此入口保留该参数但不使用。


### 4.2 信息宏

**为什么需要信息宏？** 核函数课程中数据类型通常固定（如始终使用 `half`），可以直接写死。但工程化开发中，OpDef 原型定义可声明多种数据类型（如 JSON 中 `"type": ["float16", "float"]`），同一份 Kernel 代码需要适配不同类型。

编译时，框架为每种 dtype 组合生成独立的 Kernel binary，信息宏在每个 binary 中被展开为具体的 C++ 类型常量。开发者只写一份 Kernel 代码，框架自动生成不同类型的专化版本，既避免 AI Core 上昂贵的运行时类型判断，也让每种类型组合使用对应的编译产物。

信息宏用于在 Kernel 中获取参数的类型信息，**宏名由原型定义中输入/输出的 name 字段生成**：JSON 中 `"name": "x"` → `DTYPE_X`、`"name": "y"` → `DTYPE_Y`、`"name": "z"` → `DTYPE_Z`。

- **`DTYPE_X`**：Device侧实际C++数据类型（如 `half`，dtype=float16时）
- **`ORIG_DTYPE_X`**：原始数据类型枚举宏值（如 `DT_FLOAT16`，不带 `ge::` 命名空间）
- **`FORMAT_X`**：数据格式枚举宏值（如 `FORMAT_ND`，不带 `ge::` 命名空间）

> `DTYPE_X` 和 `ORIG_DTYPE_X` 的区别：前者是 C++ 类型（`half`），后者是枚举宏值（`DT_FLOAT16`，不带命名空间）。Kernel 代码中用 `DTYPE_X` 声明变量，用 `ORIG_DTYPE_X` 做类型判断。


### 4.3 工程化 Kernel 与核函数的差异

4.1 给出了迁移清单，这里只看最关键的代码变化：核函数直调时，数据规模通常由模板参数或 `constexpr` 固定；算子工程中，数据规模来自 Host 侧 TilingData。

核函数直调实现：

```cpp
constexpr uint32_t blockLength = 2048;               // 编译时固定
xGm.SetGlobalBuffer(x + block_idx * blockLength, blockLength);
```

工程化实现：

```cpp
this->blockLength = totalLength / GetBlockNum();     // 从 TilingData 动态获取
this->tileNum = tileNum;                              // 从 TilingData 动态获取
xGm.SetGlobalBuffer((__gm__ DTYPE_X*)x + this->blockLength * GetBlockIdx(), this->blockLength);
```

因此，`CopyIn/Compute/CopyOut` 的流水逻辑可以复用，主要改造点是：把固定常量替换为成员变量，并让这些成员变量由 TilingData 初始化。


### 4.4 完整 Kernel 侧代码


In [ ]:
%%writefile Sources/06.03/custom_op/op_kernel/add_custom.cpp
#include "kernel_operator.h"
#include "add_custom_tiling.h"

using namespace AscendC;

class KernelAdd {
public:
    __aicore__ inline KernelAdd() {}
    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                                uint32_t totalLength, uint32_t tileNum);
    __aicore__ inline void Process();

private:
    __aicore__ inline void CopyIn(uint32_t loopIdx);
    __aicore__ inline void Compute(uint32_t loopIdx);
    __aicore__ inline void CopyOut(uint32_t loopIdx);

    TPipe pipe;
    TQue<TPosition::VECIN, 1> xQueue;
    TQue<TPosition::VECIN, 1> yQueue;
    TQue<TPosition::VECOUT, 1> zQueue;

    GlobalTensor<DTYPE_X> xGlobal;
    GlobalTensor<DTYPE_Y> yGlobal;
    GlobalTensor<DTYPE_Z> zGlobal;

    uint32_t blockLength;
    uint32_t tileNum;
    uint32_t tileLength;
};

__aicore__ inline void KernelAdd::Init(GM_ADDR x, GM_ADDR y, GM_ADDR z,
                                        uint32_t totalLength, uint32_t tileNum) {
    this->blockLength = totalLength / GetBlockNum();
    this->tileNum = tileNum;
    this->tileLength = this->blockLength / tileNum;

    xGlobal.SetGlobalBuffer((__gm__ DTYPE_X *)x + this->blockLength * GetBlockIdx(), this->blockLength);
    yGlobal.SetGlobalBuffer((__gm__ DTYPE_Y *)y + this->blockLength * GetBlockIdx(), this->blockLength);
    zGlobal.SetGlobalBuffer((__gm__ DTYPE_Z *)z + this->blockLength * GetBlockIdx(), this->blockLength);

    pipe.InitBuffer(xQueue, 1, this->tileLength * sizeof(DTYPE_X));
    pipe.InitBuffer(yQueue, 1, this->tileLength * sizeof(DTYPE_Y));
    pipe.InitBuffer(zQueue, 1, this->tileLength * sizeof(DTYPE_Z));
}

__aicore__ inline void KernelAdd::Process() {
    int32_t loopCount = this->tileNum;
    for (int32_t i = 0; i < loopCount; i++) {
        CopyIn(i);
        Compute(i);
        CopyOut(i);
    }
}

__aicore__ inline void KernelAdd::CopyIn(uint32_t progress) {
    LocalTensor<DTYPE_X> xLocal = xQueue.AllocTensor<DTYPE_X>();
    LocalTensor<DTYPE_Y> yLocal = yQueue.AllocTensor<DTYPE_Y>();
    DataCopy(xLocal, xGlobal[progress * this->tileLength], this->tileLength);
    DataCopy(yLocal, yGlobal[progress * this->tileLength], this->tileLength);
    xQueue.EnQue(xLocal);
    yQueue.EnQue(yLocal);
}

__aicore__ inline void KernelAdd::Compute(uint32_t progress) {
    LocalTensor<DTYPE_X> xLocal = xQueue.DeQue<DTYPE_X>();
    LocalTensor<DTYPE_Y> yLocal = yQueue.DeQue<DTYPE_Y>();
    LocalTensor<DTYPE_Z> zLocal = zQueue.AllocTensor<DTYPE_Z>();
    Add(zLocal, xLocal, yLocal, this->tileLength);
    zQueue.EnQue<DTYPE_Z>(zLocal);
    xQueue.FreeTensor(xLocal);
    yQueue.FreeTensor(yLocal);
}

__aicore__ inline void KernelAdd::CopyOut(uint32_t progress) {
    LocalTensor<DTYPE_Z> zLocal = zQueue.DeQue<DTYPE_Z>();
    DataCopy(zGlobal[progress * this->tileLength], zLocal, this->tileLength);
    zQueue.FreeTensor(zLocal);
}

extern "C" __global__ __aicore__ void add_custom(GM_ADDR x, GM_ADDR y,
                                                 GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(AddCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    KernelAdd op;
    op.Init(x, y, z, tilingData.totalLength, tilingData.tileNum);
    op.Process();
}


---

## 5. 编译部署

### 5.1 执行编译

使用 `build.sh` 一键编译：


In [ ]:
# 编译算子工程
!cd Sources/06.03/custom_op;bash build.sh

编译成功后，在 `build_out` 目录下生成 `.run` 安装包。


### 5.2 CMakePresets.json

`CMakePresets.json` 是算子工程的编译配置入口。msopgen 会根据创建工程时的参数生成该文件，CMake 配置阶段再把其中的 `cacheVariables` 注入到工程中。

先关注这些字段的作用，后续 5.3 会结合构建链路展开：

- `ASCEND_COMPUTE_UNIT`：目标 AI 处理器型号，来自 `msopgen -c` 参数。
- `vendor_name`：自定义算子所属厂商标识，影响部署目录和多 vendor 隔离。
- `ENABLE_SOURCE_PACKAGE` / `ENABLE_BINARY_PACKAGE`：控制源码包和二进制包。
- `ASCEND_CANN_PACKAGE_PATH`：CANN 软件包路径。
- `CMAKE_INSTALL_PREFIX`：安装或打包产物的输出根目录，本样例指向 `build_out`。


In [ ]:
# 查看 CMakePresets.json 配置
!cat Sources/06.03/custom_op/CMakePresets.json

### 5.3 CMake 编译组织详解

算子工程的 CMake 承担三类任务：加载算子工程编译宏、把 Host/Kernel 实现组织成不同目标，并按 RUN/SHARED/STATIC 形态安装产物。本节主线使用 RUN 模式，生成可部署的 `.run` 自定义算子安装包。

#### 5.3.1 从配置到产物的构建链路

一次构建可以按下面的链路理解：

1. `CMakePresets.json` 提供 `ASCEND_COMPUTE_UNIT`、`vendor_name`、源码/二进制打包开关等缓存变量。
2. 顶层 `CMakeLists.txt` 通过 `find_package(ASC REQUIRED)` 加载 CANN 中的 Ascend C 算子工程 CMake 模块。
3. Host 侧 `npu_op_code_gen` 读取 OpDef/Host 实现，调用 opbuild 生成 aclnn 接口代码、原型库代码、ops-info 等自动文件。
4. Host 侧 `npu_op_library` 把自动生成代码和开发者编写的 Host 代码编译为 ACLNN、GRAPH、TILING 三类目标。
5. Kernel 侧 `npu_op_kernel_sources` 描述 OpType 与 Kernel 源文件的对应关系，`npu_op_kernel_library` 登记 Kernel 源码根目录和依赖的 Tiling 库。
6. `cmake --build build_out --target binary` 编译 Kernel 二进制和配置文件，`cmake --build build_out --target package` 生成 RUN 安装包。

开发者通常只改 Host 实现、Kernel 实现、TilingData 定义和少量配置项；CMakeLists 更多是在声明工程结构。

#### 5.3.2 CMakeLists.txt 层级结构

msopgen 默认生成的工程采用“按 Host/Kernel 划分”的三层结构：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">层级</th>
      <th align="left">文件位置</th>
      <th align="left">职责</th>
      <th align="left">开发者通常改什么</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>根目录</td>
      <td><code>CMakeLists.txt</code></td>
      <td>加载 ASC CMake 模块，定义 package 类型和输出位置，添加子目录</td>
      <td>少量修改 vendor、package 类型、子目录组织</td>
    </tr>
    <tr>
      <td>Host 侧</td>
      <td><code>op_host/CMakeLists.txt</code></td>
      <td>执行代码生成，编译 aclnn 接口库、算子原型库、Tiling 库</td>
      <td>多算子时调整 Host 源文件收集方式</td>
    </tr>
    <tr>
      <td>Kernel 侧</td>
      <td><code>op_kernel/CMakeLists.txt</code></td>
      <td>登记 Kernel 源码，编译 Kernel 二进制或打包源码</td>
      <td>Kernel 文件不按默认命名时，显式配置 OpType 与文件映射</td>
    </tr>
  </tbody>
</table>

#### 5.3.3 编译产物与来源

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">产物</th>
      <th align="left">主要来源</th>
      <th align="left">用途</th>
      <th align="left">RUN 包中的典型位置</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>aclnn_*.h</code></td>
      <td>由 <code>npu_op_code_gen</code> 自动生成</td>
      <td>单算子 API 头文件，供验证工程或业务程序 include</td>
      <td><code>op_api/include</code></td>
    </tr>
    <tr>
      <td><code>libcust_opapi.so</code></td>
      <td>自动生成的 aclnn 源码编译得到</td>
      <td>aclnn 两段式接口实现库</td>
      <td><code>op_api/lib</code></td>
    </tr>
    <tr>
      <td><code>libcust_opsproto_rt2.0.so</code> / <code>op_proto.h</code></td>
      <td>Host 侧 OpDef、InferShape 与自动生成原型代码</td>
      <td>算子原型定义库，支撑入图和框架识别算子</td>
      <td><code>op_proto</code></td>
    </tr>
    <tr>
      <td><code>libcust_opmaster_rt2.0.so</code> / <code>liboptiling.so</code></td>
      <td>Host 侧 TilingFunc 和 Tiling 相关代码</td>
      <td>运行时根据输入 shape、dtype、format 计算 Tiling 参数</td>
      <td><code>op_impl/ai_core/tbe/op_tiling</code></td>
    </tr>
    <tr>
      <td><code>*.o</code> 与 Kernel 配置 <code>*.json</code></td>
      <td>Kernel 侧 Ascend C 源码编译得到</td>
      <td>二进制发布场景下，运行时直接匹配并执行 Kernel binary</td>
      <td><code>op_impl/ai_core/tbe/kernel/&lt;soc&gt;</code></td>
    </tr>
    <tr>
      <td>Kernel 源码</td>
      <td>开发者编写的 <code>op_kernel/*.cpp</code> 和相关头文件</td>
      <td>源码发布或在线编译场景使用</td>
      <td><code>op_impl/ai_core/tbe/&lt;vendor&gt;_impl/dynamic</code></td>
    </tr>
  </tbody>
</table>

`ENABLE_SOURCE_PACKAGE` 控制源码发布，`ENABLE_BINARY_PACKAGE` 控制二进制发布。aclnn 单算子调用通常依赖二进制产物；需要同时保留源码以支持在线编译时，可以两个开关都开启。

#### 5.3.4 顶层 CMakeLists.txt：定义工程和 package

样例顶层文件只负责加载 CMake 模块、定义 RUN 包、添加子目录：

```cmake
cmake_minimum_required(VERSION 3.16.0)
project(opp)
find_package(ASC REQUIRED)
set(package_name ${vendor_name})

npu_op_package(${package_name}
    TYPE RUN
    CONFIG
        INSTALL_PATH ${CMAKE_BINARY_DIR}/
)

if(EXISTS ${CMAKE_CURRENT_SOURCE_DIR}/framework)
    add_subdirectory(framework)
endif()
if(EXISTS ${CMAKE_CURRENT_SOURCE_DIR}/op_host)
    add_subdirectory(op_host)
endif()
if(EXISTS ${CMAKE_CURRENT_SOURCE_DIR}/op_kernel)
    add_subdirectory(op_kernel)
endif()
```

关键点：

- `find_package(ASC REQUIRED)` 会加载 CANN 软件包中的算子工程 CMake 函数，例如 `npu_op_package`、`npu_op_library`、`npu_op_kernel_sources`。
- `set(package_name ${vendor_name})` 将 vendor 名称作为 package 名称；RUN 模式下它会参与部署目录组织。
- `TYPE RUN` 表示生成 `.run` 部署包。`SHARED` 和 `STATIC` 分别用于算子动态库、静态库形态，后续构建目标也会从 `package` 变为 `install`。
- `add_subdirectory(op_host)` 要先于 `add_subdirectory(op_kernel)`。Kernel 侧的源码登记和编译选项依赖 Host 侧 `npu_op_code_gen` 已经生成的自动文件路径。

#### 5.3.5 Host 侧 CMakeLists.txt：代码生成和 Host 库

Host 侧 CMakeLists 的核心是先生成自动代码，再按用途拆成三类库：

```cmake
aux_source_directory(${CMAKE_CURRENT_SOURCE_DIR} ops_srcs)
npu_op_code_gen(
    SRC ${ops_srcs}
    PACKAGE ${package_name}
    OUT_DIR ${ASCEND_AUTOGEN_PATH}
)

file(GLOB autogen_aclnn_src ${ASCEND_AUTOGEN_PATH}/aclnn_*.cpp)
set_source_files_properties(${autogen_aclnn_src} PROPERTIES GENERATED TRUE)
npu_op_library(cust_opapi ACLNN
    ${autogen_aclnn_src}
)

file(GLOB proto_src ${ASCEND_AUTOGEN_PATH}/op_proto.cc)
set_source_files_properties(${proto_src} PROPERTIES GENERATED TRUE)
npu_op_library(cust_op_proto GRAPH
    ${ops_srcs}
    ${proto_src}
)

file(GLOB fallback_src ${ASCEND_AUTOGEN_PATH}/fallback_*.cpp)
set_source_files_properties(${fallback_src} PROPERTIES GENERATED TRUE)
npu_op_library(cust_optiling TILING
    ${ops_srcs}
    ${fallback_src}
)

npu_op_package_add(${package_name}
    LIBRARY
        cust_optiling
        cust_opapi
        cust_op_proto
)
```

各步骤的含义：

- `aux_source_directory` 收集 Host 侧实现文件。本节只有一个 `add_custom.cpp`，多算子工程可以改用 `file(GLOB ...)` 或显式列出多个 Host 源文件。
- `npu_op_code_gen` 会根据 Host 侧 OpDef 生成 `aclnn_*.cpp`、`aclnn_*.h`、`op_proto.cc`、ops-info 等文件，并把生成目录记录到 CMake 全局属性中。
- `set_source_files_properties(... GENERATED TRUE)` 告诉 CMake 这些源文件在配置之后由构建流程生成，不要求它们在配置初期已经存在。
- `npu_op_library(cust_opapi ACLNN ...)` 编译单算子 API 调用库，验证工程链接的就是这个库。
- `npu_op_library(cust_op_proto GRAPH ...)` 编译算子原型定义库，入图和框架调用场景需要它。
- `npu_op_library(cust_optiling TILING ...)` 编译 Tiling 库，Kernel 二进制编译和运行时执行都会依赖它。
- `npu_op_package_add(... LIBRARY ...)` 把这些 Host 侧目标加入 package，后续 install/package 阶段会把它们放到 RUN 包的约定目录。

#### 5.3.6 Kernel 侧 CMakeLists.txt：源码登记和 Kernel 库

样例 Kernel 文件位于 `op_kernel/` 根目录，因此可以用 `KERNEL_DIR ./` 描述整个目录：

```cmake
npu_op_kernel_sources(ascendc_kernels
    KERNEL_DIR ./
)

npu_op_kernel_library(ascendc_kernels
    SRC_BASE ${CMAKE_CURRENT_SOURCE_DIR}/
    TILING_LIBRARY cust_optiling
)

npu_op_package_add(${package_name}
    LIBRARY ascendc_kernels
)
```

这里有三个容易混淆的点：

- `npu_op_kernel_sources` 不是普通的 `target_sources`。它会把 OpType、Kernel 文件、Kernel 目录、芯片型号这些信息写入自动生成配置，后续 Kernel 编译脚本再按这些配置逐个编译。
- `SRC_BASE` 是 Kernel 源码根目录，`KERNEL_DIR` 是相对该根目录的源码子目录。源码发布场景会按这个根目录收集并打包 Kernel 相关文件。
- `TILING_LIBRARY cust_optiling` 建立 Kernel 编译对 Tiling 库的依赖。编译 Kernel binary 时需要结合 ops-info、Tiling 库和目标芯片型号生成二进制与配置文件。

如果 Kernel 文件名和 OpType 的默认转换规则不一致，或者一个目录下包含多个算子，建议显式写出映射关系：

```cmake
npu_op_kernel_sources(ascendc_kernels
    OP_TYPE AddCustom
    KERNEL_DIR add_custom
    KERNEL_FILE add_custom_kernel.cpp
)

npu_op_kernel_sources(ascendc_kernels
    OP_TYPE LeakyReluCustom
    KERNEL_DIR leaky_relu_custom
    KERNEL_FILE leaky_relu_custom_kernel.cpp
)
```

如果只想给某个算子或某个芯片型号追加 Kernel 编译选项，可以使用 `npu_op_kernel_options`：

```cmake
npu_op_kernel_options(ascendc_kernels ALL OPTIONS -g)
npu_op_kernel_options(ascendc_kernels AddCustom COMPUTE_UNIT ascend910b OPTIONS -DDEBUG_MODE=1)
```

Host 侧编译选项则继续使用 CMake 原生接口：

```cmake
target_compile_options(cust_opapi PRIVATE -fvisibility=hidden)
target_compile_options(cust_optiling PRIVATE -fvisibility=hidden)
```

#### 5.3.7 常用 CMake 配置项

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">配置项</th>
      <th align="left">常见来源</th>
      <th align="left">作用</th>
      <th align="left">本课程建议</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>vendor_name</code></td>
      <td><code>CMakePresets.json</code> 或 <code>set()</code></td>
      <td>标识自定义算子所属厂商，影响部署目录和优先级</td>
      <td>练习可用默认 <code>customize</code>，正式工程建议使用独立名称</td>
    </tr>
    <tr>
      <td><code>ASCEND_COMPUTE_UNIT</code></td>
      <td><code>msopgen -c</code> 写入，或手动配置</td>
      <td>指定要编译的 AI 处理器型号，可配置多个型号</td>
      <td>保持与运行环境一致；多型号编译时逐个确认支持情况</td>
    </tr>
    <tr>
      <td><code>ASCEND_AUTOGEN_PATH</code></td>
      <td>CANN CMake 模块默认设置</td>
      <td>保存 <code>npu_op_code_gen</code> 生成的 aclnn、proto、ops-info 等文件</td>
      <td>通常不需要修改，调试时可进入 <code>build_out/autogen</code> 查看</td>
    </tr>
    <tr>
      <td><code>ENABLE_SOURCE_PACKAGE</code></td>
      <td><code>CMakePresets.json</code> 或 <code>npu_op_package CONFIG</code></td>
      <td>控制是否打包 Kernel 源码</td>
      <td>入门阶段可保持开启，便于观察包结构</td>
    </tr>
    <tr>
      <td><code>ENABLE_BINARY_PACKAGE</code></td>
      <td><code>CMakePresets.json</code> 或 <code>npu_op_package CONFIG</code></td>
      <td>控制是否编译并打包 Kernel 二进制</td>
      <td>aclnn 调用验证需要开启；SHARED/STATIC 模式也要求开启</td>
    </tr>
    <tr>
      <td><code>ASCEND_CANN_PACKAGE_PATH</code></td>
      <td>环境变量或 <code>CMakePresets.json</code></td>
      <td>定位 CANN 安装路径、opbuild、编译脚本和依赖库</td>
      <td>保持与当前 source 的 CANN 环境一致</td>
    </tr>
    <tr>
      <td><code>ASCEND_SKIP_FAILED_COMPUTE_UNIT</code></td>
      <td>手动配置</td>
      <td>多芯片型号编译时，控制部分型号失败是否跳过</td>
      <td>多型号批量构建时再考虑开启，单型号调试建议保持关闭</td>
    </tr>
  </tbody>
</table>

同一个开关既出现在 `CMakePresets.json` 又出现在 `npu_op_package(... CONFIG ...)` 时，package 级配置会作用到当前 package 的打包行为。入门阶段建议只在 `CMakePresets.json` 中改芯片型号和 vendor 名称。

#### 5.3.8 多算子和多 vendor 时如何组织

默认的 Host/Kernel 分层方式适合单算子入门工程，也适合少量算子共用同一套 package。多算子工程中，常见做法是继续保留 `op_host/` 和 `op_kernel/` 两层，再在内部按算子建子目录：

```text
op_host/
├── add_custom/add_custom_host.cpp
└── leaky_relu_custom/leaky_relu_custom_host.cpp

op_kernel/
├── add_custom/add_custom_kernel.cpp
└── leaky_relu_custom/leaky_relu_custom_kernel.cpp
```

此时 Host 侧显式收集多个源文件，Kernel 侧为每个 OpType 调用一次 `npu_op_kernel_sources`。这样新增算子时，只需要新增源文件目录并补充一次映射，原有算子的构建规则不受影响。

如果不同 vendor 的算子包需要在同一个顶层工程中并行构建，更适合把每个 vendor 当作独立子工程，用 `ExternalProject_Add` 分别传入 `vendor_name`、`ASCEND_COMPUTE_UNIT` 和安装目录。这样每个子工程都会生成独立 RUN 包，避免包名、部署路径、OpType 优先级互相干扰。

#### 5.3.9 构建命令与版本注意

RUN 包构建分两步：先生成二进制相关产物，再生成 `.run` 包。

```bash
cmake -S . -B build_out --preset=default
cmake --build build_out --target binary -j$(nproc)
cmake --build build_out --target package -j$(nproc)
```

如果 package 类型改为 `SHARED` 或 `STATIC`，最后一步通常改为 `install`：

```bash
cmake --build build_out --target binary -j$(nproc)
cmake --build build_out --target install -j$(nproc)
```

需要注意两点：

- `cmake_minimum_required(VERSION 3.16.0)` 表示这些 CMakeLists 语法本身的最低要求；`--preset` 依赖 CMakePresets 命令行能力，旧版 CMake 不支持时需要把 preset 解析成普通 CMake 参数。
- 本节使用 `build.sh` 封装上述命令。理解 5.3 的目的不是手写所有 CMake，而是在编译失败、增加算子、改变发布形态或调整芯片型号时，知道该检查哪一层配置。


### 5.4 部署

5.3 完成编译后，`build_out` 目录中会生成自定义算子 `.run` 安装包。部署就是执行该安装包，把 `op_api`、`op_proto`、`op_impl` 等目录安装到 CANN 可发现的位置。

```bash
./custom_opp_*.run --install-path=<path>
```

- 不指定 `--install-path` 时，默认安装到 `$ASCEND_OPP_PATH/vendors/<vendor_name>`，通常随当前 CANN 环境自动生效。
- 指定目录安装后，需要执行 `source <path>/vendors/<vendor_name>/bin/set_env.bash`，让运行时能找到该自定义算子包。
- 多个自定义算子包包含相同 OpType 时，加载优先级会影响实际命中的实现；部署优先级细节见扩展阅读 E5。


In [ ]:
# 部署算子包到指定目录
!./Sources/06.03/custom_op/build_out/custom_opp*.run --install-path=${HOME}/

---

## 6. 调用验证

算子部署完成后，调用方链接 `libcust_opapi.so` 并 include 自动生成的 `aclnn_*.h`，即可在应用程序中调用自定义算子。

### 6.1 两段式接口

aclnn 单算子API采用**两段式设计**，形如：

```cpp
aclnnStatus aclnnXxxGetWorkspaceSize(const aclTensor *src, ..., aclTensor *out,
                                      uint64_t *workspaceSize, aclOpExecutor **executor);
aclnnStatus aclnnXxx(void *workspace, uint64_t workspaceSize,
                      aclOpExecutor *executor, aclrtStream stream);
```

- `aclnnXxxGetWorkspaceSize`（第一段）：计算本次API调用所需的 workspace 内存大小，匹配 kernel binary，生成 executor
- `aclnnXxx`（第二段）：执行算子计算

参数命名规则：可选输入增加 <code>Optional</code> 后缀；输入输出同名时保留 input 参数并以 <code>Ref</code> 作后缀；单个输出命名为 <code>out</code>，多个输出以 <code>Out</code> 作后缀；属性参数位于输入输出之间。Xxx 为算子原型注册时的 OpType。

### 6.2 最小调用流程

完整的 aclnn 调用包含 10 个步骤：


<img src="./images/aclnn_single_operator_call_flow.png" alt="aclnn single operator call flow" width="700px" />


### 6.2.1 步骤代码对照

以下将 10 步流程与实际调用代码逐步对照（完整代码见 `test/main.cpp`）：

**步骤 1-2：ACL 初始化 + 设置 Device + 创建 Stream**

```cpp
auto ret = aclInit(nullptr);                  // 初始化 ACL 运行时
ret = aclrtSetDevice(deviceId);               // 指定 NPU 设备（deviceId = 0）
ret = aclrtCreateStream(&stream);             // 创建执行流
```

**步骤 3-4：创建 aclTensor + 申请 Device 内存**

`CreateAclTensor` 封装了内存分配、Host→Device 拷贝与 aclTensor 创建：

```cpp
// 输入 shape: {8, 2048}，dtype: ACL_FLOAT16
// x 全 1.0，y 全 2.0，z 全 0.0（期望输出全 3.0）
std::vector<int64_t> shape = {8, 2048};
std::vector<aclFloat16> xHost(8 * 2048, aclFloatToFloat16(1.0));
std::vector<aclFloat16> yHost(8 * 2048, aclFloatToFloat16(2.0));
std::vector<aclFloat16> zHost(8 * 2048, aclFloatToFloat16(0.0));

// CreateAclTensor 内部依次调用：
//   aclrtMalloc()  → Device 侧分配内存
//   aclrtMemcpy()  → Host 数据拷贝到 Device
//   aclCreateTensor() → 创建 aclTensor 对象
ret = CreateAclTensor(xHost, shape, &xDevAddr, ACL_FLOAT16, &inputX);
ret = CreateAclTensor(yHost, shape, &yDevAddr, ACL_FLOAT16, &inputY);
ret = CreateAclTensor(zHost, shape, &zDevAddr, ACL_FLOAT16, &outputZ);
```

**步骤 5：第一段调用**

```cpp
uint64_t workspaceSize = 0;
aclOpExecutor *executor;
ret = aclnnAddCustomGetWorkspaceSize(inputX, inputY, outputZ,
                                     &workspaceSize, &executor);
```

**步骤 6：申请 workspace**（workspaceSize=0 时可跳过）

```cpp
void *workspaceAddr = nullptr;
if (workspaceSize > 0) {
    ret = aclrtMalloc(&workspaceAddr, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST);
}
```

**步骤 7：第二段调用**

```cpp
ret = aclnnAddCustom(workspaceAddr, workspaceSize, executor, stream);
```

**步骤 8：同步 + 拷回结果**

```cpp
ret = aclrtSynchronizeStream(stream);         // 等待算子执行完成

std::vector<aclFloat16> result(8 * 2048, 0);
ret = aclrtMemcpy(result.data(), result.size() * sizeof(aclFloat16),
                  zDevAddr, 8 * 2048 * sizeof(aclFloat16),
                  ACL_MEMCPY_DEVICE_TO_HOST);  // Device → Host
```

**步骤 9：释放资源**

```cpp
aclDestroyTensor(inputX);  aclDestroyTensor(inputY);  aclDestroyTensor(outputZ);
aclrtFree(xDevAddr);       aclrtFree(yDevAddr);       aclrtFree(zDevAddr);
if (workspaceAddr) aclrtFree(workspaceAddr);
aclrtDestroyStream(stream);
aclrtResetDevice(deviceId);
```

**步骤 10：去初始化**

```cpp
aclFinalize();                              // 去初始化 ACL 运行时
```

### 6.3 运行验证

运行预置的调用程序验证算子功能，该程序按上述流程编写：初始化→创建 aclTensor→申请内存→两段调用→拷回验证。


In [ ]:
# 复制测试代码
!cp -r src/custom_op/test Sources/06.03/test


In [ ]:
# 编译测试代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/06.03/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/06.03/execute_add_op


In [ ]:
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/06.03/execute_add_op


调用程序正常执行后会有如下打印：

```
result is:
3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0 3.0
test pass
```


## 本节小结

本节完成了一条**设计→实现→编译部署→调用验证**的工程化链路：

- **设计**：从公式识别输入输出→JSON 原型定义→msopgen 创建工程→理解 OpDef
- **实现**：标准C++ 定义 TilingData→TilingFunc 计算参数并设置 BlockDim/workspace→Kernel侧 REGISTER_TILING_DEFAULT + GET_TILING_DATA 获取参数→DTYPE_X/Y/Z 适配类型
- **编译部署**：`build.sh` → `.run` 包 → `set_env.bash`
- **调用验证**：aclnn 两段式接口（GetWorkspaceSize + Execute）验证

进阶内容（属性传递、多分支策略等）详见课后练习之后的扩展阅读。


---

## 课后练习

请根据已提供的 SubCustom 算子原型 json 文件，完成算子工程创建，并完成功能实现。


算子原型json文件：


In [ ]:
%%writefile Sources/06.03/sub_custom.json
[{
    "op": "SubCustom",
    "input_desc": [{
            "name": "x",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        },
        {
            "name": "y",
            "param_type": "required",
            "format": ["ND", "ND"],
            "type": ["float16", "float"]
        }
    ],
    "output_desc": [{
        "name": "z",
        "param_type": "required",
        "format": ["ND", "ND"],
        "type": ["float16", "float"]
    }]
}]


使用 msopgen 生成算子工程：


In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/06.03/custom_op

# 使用msopgen工具创建算子工程
!msopgen gen -i Sources/06.03/sub_custom.json -c ai_core-ascend910b1 -lan cpp -out Sources/06.03/custom_op


修改 host侧实现：


In [ ]:
%%writefile Sources/06.03/custom_op/op_host/sub_custom.cpp

#include "../op_kernel/sub_custom_tiling.h"
#include "register/op_def_registry.h"

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  SubCustomTilingData *tiling = context->GetTilingData<SubCustomTilingData>();
  const gert::StorageShape* x1_shape = context->GetInputShape(0);
  int32_t data_sz = 1;
  for (int i = 0; i < x1_shape->GetStorageShape().GetDimNum(); i++)
    data_sz *= x1_shape->GetStorageShape().GetDim(i);
  tiling->totalLength = data_sz;
  tiling->tileNum = 1;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class SubCustom : public OpDef {
public:
    explicit SubCustom(const char* name) : OpDef(name)
    {
        this->Input("x")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Input("y")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});
        this->Output("z")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16, ge::DT_FLOAT})
            .Format({ge::FORMAT_ND, ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend950");
    }
};
OP_ADD(SubCustom);
}


修改 Tiling 结构体定义：


In [ ]:
%%writefile Sources/06.03/custom_op/op_kernel/sub_custom_tiling.h
#ifndef SUB_CUSTOM_TILING_H
#define SUB_CUSTOM_TILING_H
#include <cstdint>

struct SubCustomTilingData {
    uint32_t totalLength;
    uint32_t tileNum;
};

#endif


修改 kernel 实现：


In [ ]:
%%writefile Sources/06.03/custom_op/op_kernel/sub_custom.cpp
#include "kernel_operator.h"
#include "sub_custom_tiling.h"

extern "C" __global__ __aicore__ void sub_custom(GM_ADDR x, GM_ADDR y, GM_ADDR z, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(SubCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    // TODO: user kernel impl
}


注意：这里的 `TODO` 是占位，先按 AddCustom Kernel 的 `Init -> Process -> CopyIn -> Compute -> CopyOut` 模式补全 SubCustom 实现；未补全前不要继续执行后续编译和测试单元。

修改完成后，尝试运行已准备好的输入Shape为[8, 2048]，数据类型为half的测试用例，检查算子是否符合预期。


In [ ]:
# 编译部署修改后的算子
!cd Sources/06.03/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/


In [ ]:
# 清除已有的测试代码
!rm -rf Sources/06.03/test;
# 复制已有的测试代码
!cp -r src/custom_op/test_sub Sources/06.03/test;
# 编译测试代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/06.03/test/main.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/06.03/execute_sub_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/06.03/execute_sub_op


预期窗口输出：

```
result is:
-1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0 -1.0
test pass
```


---

## 参考答案

执行以下代码查看答案：


host侧实现：


In [ ]:
!cat ./answer/06.03_answer/op_host/sub_custom.cpp


Tiling 结构体定义：


In [ ]:
!cat ./answer/06.03_answer/op_kernel/sub_custom_tiling.h


Kernel侧实现：


In [ ]:
!cat ./answer/06.03_answer/op_kernel/sub_custom.cpp


---

## 扩展阅读

以下内容属于进阶专题，按需深入阅读。

### E1. 属性传递与 OpDef 属性定义

当算子有属性（如 `z = x + alpha * y` 中的 `alpha`）时，需要两步处理：

**1. OpDef 中注册属性**：使用 `.Int()` / `.Float()` / `.Bool()` / `.String()` / `.ListInt()` / `.ListFloat()` / `.ListBool()` 等方法。

属性注册方式与 aclnn API 参数类型的常见对应关系：`.Int()` → `int64_t`，`.Float()` → `float`，`.Bool()` → `bool`，`.String()` → `string`；`.ListInt()` / `.ListFloat()` / `.ListBool()` 分别对应 `vector<int64_t>` / `vector<float>` / `vector<bool>`。

**2. TilingFunc 中获取属性并传递到 Kernel侧**：

```cpp
auto attrs = context->GetAttrs();
float *alpha = attrs->GetAttrPointer<float>("alpha");  // 按名称
// float *alpha = attrs->GetAttrPointer<float>(0);     // 按索引
```

然后在 TilingData 中增加属性字段，并在 TilingFunc 中赋值传递到 Kernel侧。

**设计原则：TilingData 只传 Kernel侧需要的标量值。**


### E2. 多分支策略（TilingKey 与 Kernel 模板编程）

当一个算子在不同场景下需要走不同的算法逻辑时，可以将 Kernel 代码拆分为多个独立入口函数以减少 icache miss，或通过模板参数化提升多分支场景的可读性。

**TilingKey 分支策略**：使用数字标识不同的 Kernel 实现分支，核心价值在于**减少 icache miss 和 scalar 耗时**。`TILING_KEY_IS` 是编译期常量折叠，编译器会为每个 TilingKey 生成独立的 Kernel binary。

```cpp
// Host侧设置 TilingKey
if (totalLength <= 1024) {
    context->SetTilingKey(1);  // 小数据量：单块处理
} else {
    context->SetTilingKey(2);  // 大数据量：多块切分处理
}

// Kernel侧使用 TILING_KEY_IS
if (TILING_KEY_IS(1)) {
    op.ProcessSmall();   // 编译期常量折叠，未命中分支完全移除
} else if (TILING_KEY_IS(2)) {
    op.ProcessLarge();
}
```

<img src="./images/tilingkey.png" alt="TilingKey" />

**选择准则**：
1. Kernel 代码需要在 <<<>>> 和 aclnn 场景共用时，使用 Kernel 模板方式
2. 分支较多、需要提升可读性时，使用 Kernel 模板方式
3. 其他场景，按需选择 TilingKey 或 Kernel 模板


### E4. 运行时加载机制

6.1 已从调用方视角介绍两段式接口，这里补充它对调用流程和性能分析的影响：

**第一阶段 GetWorkspaceSize**：根据输入 Tensor 描述、属性和算子原型完成执行前准备，并返回本次执行需要的 workspace 大小。TilingFunc 在这一阶段参与计算 TilingData、TilingKey、BlockDim 和 workspace 信息。

**第二阶段 Execute**：调用方分配 workspace 后，将输入/输出 Tensor、workspace、executor 和 stream 传入执行接口，由执行接口提交 Kernel 执行。

**为什么分两阶段**：将 workspace 内存分配与 kernel 执行解耦，用户可以在拿到多个算子的 workspace 需求后统一规划内存，减少 aclrtMalloc 调用次数和内存碎片。

**影响运行时性能的因素**：

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">类别</th>
      <th align="left">具体内容</th>
      <th align="left">影响</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>编译产物形态</td>
      <td>TilingKey 数量、compute_unit 范围、ENABLE_BINARY_PACKAGE</td>
      <td>执行准备与加载耗时</td>
    </tr>
    <tr>
      <td>Host侧开销</td>
      <td>TilingFunc 复杂度、TilingData 字段数、SetBlockDim/workspace</td>
      <td>GetWorkspaceSize 耗时</td>
    </tr>
    <tr>
      <td>Kernel侧表现</td>
      <td>TILING_KEY_IS vs if-else、模板参数、单双buffer、tileNum 粒度</td>
      <td>执行性能</td>
    </tr>
  </tbody>
</table>

### E5. 部署细节与优先级

#### 默认安装与指定目录安装

**默认安装**（不指定 --install-path）：安装到 `$ASCEND_OPP_PATH/vendors/<vendor_name>` 目录。

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">安装方式</th>
      <th align="left">安装路径</th>
      <th align="left">是否需要额外配置</th>
      <th align="left">优先级</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>默认安装</td>
      <td><code>${INSTALL_DIR}/opp/vendors/<vendor_name></code></td>
      <td>否，自动生效</td>
      <td>较低</td>
    </tr>
    <tr>
      <td>指定目录安装</td>
      <td><code><path>/vendors/<vendor_name></code></td>
      <td>是，需要 source 环境变量</td>
      <td>较高</td>
    </tr>
  </tbody>
</table>

> - 默认安装因权限不足失败时，可使用指定目录安装
> - 指定目录安装后，需执行 `source <path>/vendors/<vendor_name>/bin/set_env.bash`

#### 部署后的目录结构

```
├── opp/vendors/vendor_name1
│   ├── framework     //自定义算子插件库
│   ├── op_api        // aclnn_*.h + libcust_opapi.so
│   ├── op_impl       // Kernel二进制 + Tiling动态库
│   └── op_proto      // 算子原型动态库
```

#### 配置自定义算子优先级

多算子包共存时，若存在相同 OpType 的自定义算子，以优先级高的算子包为准。
- **默认安装场景**：配置 `opp/vendors/config.ini`：`load_priority=vendor_name1,vendor_name2,vendor_name3`（从高到低）
- **指定目录安装场景**：`set_env.bash` 脚本执行顺序越靠后，优先级越高

#### 卸载与更新

- **卸载**：执行 `bash <安装路径>/vendors/<vendor_name>/uninstall.sh`
- **更新（默认安装）**：重新执行安装命令，自动覆盖旧版本
- **更新（指定目录）**：先卸载旧版本，再安装新版本

### E6. 编译调试与交叉编译

**编译调试技巧**：

- **查看实际编译命令**：保留中间产物，设置 `ASCENDC_BUILD_LOG_DIR` 存储编译日志
- **选择性编译**：开发调试时可通过 `--tiling_key` 仅编译指定分支，加速迭代
- **常见问题**：CMake 版本不满足、芯片型号未配置、glibc 版本不兼容等

**交叉编译**：

编译平台与运行平台架构不同时（如 x86_64 编译，部署到 aarch64），可使用交叉编译。
**CMakePresets.json 配置**：

```json
"ENABLE_CROSS_COMPILE": { "type": "BOOL", "value": "True" },
"CMAKE_CROSS_PLATFORM_COMPILER": { "type": "PATH", "value": "/usr/bin/aarch64-linux-gnu-g++" }
```

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">当前平台</th>
      <th align="left">目标平台</th>
      <th align="left">安装命令</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>x86_64</td>
      <td>aarch64</td>
      <td><code>sudo apt-get install -y g++-aarch64-linux-gnu</code></td>
    </tr>
    <tr>
      <td>aarch64</td>
      <td>x86_64</td>
      <td><code>sudo apt-get install -y g++-x86-64-linux-gnu</code></td>
    </tr>
  </tbody>
</table>

### E7. 验证工程的 CMakeLists.txt 配置

调用示例也可以用 CMakeLists.txt 管理。下面匹配前文的指定目录安装方式：算子包位于 `${HOME}/vendors/customize`；默认安装时只需把 `CUSTOM_OPP_ROOT` 改成对应的 `vendors/customize` 目录。

```cmake
cmake_minimum_required(VERSION 3.16)
project(AddCustomTest LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_SKIP_RPATH TRUE)
set(CUSTOM_OPP_ROOT "$ENV{HOME}/vendors/customize")

add_executable(execute_add_op main.cpp)

# 头文件搜索路径：CANN基础头文件 + 算子API头文件
target_include_directories(execute_add_op PRIVATE
    $ENV{ASCEND_HOME_PATH}/include
    ${CUSTOM_OPP_ROOT}/op_api/include
)

# 库文件搜索路径
target_link_directories(execute_add_op PRIVATE
    $ENV{ASCEND_HOME_PATH}/lib64
    ${CUSTOM_OPP_ROOT}/op_api/lib
)

# 链接依赖库
target_link_libraries(execute_add_op PRIVATE
    cust_opapi
    nnopbase
    acl_rt
)
```


### E8. 高阶 API 配套 Tiling（宏定义方式与 TCubeTiling）

高阶 API（如 Matmul 类接口）通常会配套专用 Tiling 结构体，例如 `TCubeTiling`。

采用宏定义方式组织 TilingData 时，可通过 `BEGIN_TILING_DATA_DEF` / `TILING_DATA_FIELD_DEF_STRUCT` / `END_TILING_DATA_DEF` 等宏把 `TCubeTiling` 作为字段放入自定义 TilingData；TilingFunc 调用 `MultiCoreMatmulTiling` 等接口完成计算后，将结果写入该字段。

**宏定义方式与标准C++方式的取舍**：
- 宏定义方式：适合维护已有宏方式工程，或在高阶 API 示例代码中沿用 `TCubeTiling` 字段组织方式
- 标准C++方式：普通算子可直接定义 POD 结构体；使用高阶 API 时，也可以在标准 C++ TilingData 中引用 `AscendC::tiling` 命名空间下的预定义 Tiling 结构体

> 本课程主线使用标准 C++ TilingData；遇到 Matmul 等高阶 API 算子时，再根据所选接口示例选择宏方式或标准 C++ 方式。


### 术语速查

<table style="margin-left: 0; margin-right: auto; width: auto; border: 1px solid #ddd; text-align: left;">
  <thead>
    <tr>
      <th align="left">术语</th>
      <th align="left">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>msopgen</td>
      <td>CANN 软件包自带的算子工程骨架生成工具</td>
    </tr>
    <tr>
      <td>OpDef</td>
      <td>算子原型定义类，继承自 <code>ge::OpDef</code></td>
    </tr>
    <tr>
      <td>OP_ADD</td>
      <td>算子注册宏</td>
    </tr>
    <tr>
      <td>ParamType</td>
      <td>输入输出参数类型：REQUIRED/OPTIONAL/DYNAMIC</td>
    </tr>
    <tr>
      <td>TilingData</td>
      <td>Host侧向 Kernel侧传递的分块参数数据结构</td>
    </tr>
    <tr>
      <td>TilingFunc</td>
      <td>Host侧计算 Tiling 参数的入口函数</td>
    </tr>
    <tr>
      <td>BlockDim</td>
      <td>逻辑 Block 数，由 TilingFunc 中 <code>SetBlockDim()</code> 设置</td>
    </tr>
    <tr>
      <td>workspace</td>
      <td>算子执行所需的额外 Device 侧内存，本例设为 0</td>
    </tr>
    <tr>
      <td>REGISTER_TILING_DEFAULT</td>
      <td>Kernel侧注册标准C++ TilingData结构体的宏</td>
    </tr>
    <tr>
      <td>GET_TILING_DATA</td>
      <td>Kernel侧反序列化 TilingData 的宏</td>
    </tr>
    <tr>
      <td>DTYPE_X</td>
      <td>Kernel侧信息宏，由输入输出 name 生成，获取参数的实际C++类型</td>
    </tr>
    <tr>
      <td>ORIG_DTYPE_X</td>
      <td>Kernel侧信息宏，获取参数的原始枚举值</td>
    </tr>
    <tr>
      <td>FORMAT_X</td>
      <td>Kernel侧信息宏，获取参数的数据格式</td>
    </tr>
    <tr>
      <td>InferShape</td>
      <td>输出 shape 推导函数，可用 Follow 接口简化</td>
    </tr>
    <tr>
      <td>InferDataType</td>
      <td>输出 datatype 推导函数</td>
    </tr>
    <tr>
      <td>aclnn</td>
      <td>Ascend C 算子的单算子API调用接口</td>
    </tr>
    <tr>
      <td>两段式接口</td>
      <td>aclnnXxxGetWorkspaceSize + aclnnXxx</td>
    </tr>
    <tr>
      <td>.run包</td>
      <td>算子部署包格式</td>
    </tr>
    <tr>
      <td>CMakePresets.json</td>
      <td>编译预设配置文件（ASCEND_COMPUTE_UNIT + vendor_name）</td>
    </tr>
    <tr>
      <td>GetOriginShape / GetStorageShape</td>
      <td>前者返回逻辑 shape，后者返回物理存储 shape；ND 格式时两者一致</td>
    </tr>
  </tbody>
</table>
